# Análise de E-Commerce Brasileiro (Dataset Olist)
### Engenharia e Análise de Dados com DuckDB e SQL

Este projeto realiza a ingestão, modelagem analítica e extração de insights sobre pedidos, prazos de entrega, categorias mais vendidas e faturamento do marketplace Olist.

In [1]:
from pathlib import Path
import duckdb

BASE_DIR = Path().resolve().parent
db_path = BASE_DIR / "database" / "olist.duckdb"

# Conecta via DuckDB
con = duckdb.connect(str(db_path), read_only=True)

# Passa a conexão existente para a extensão SQL
%reload_ext sql
%sql con
%config SqlMagic.autolimit = 50

# Para exibir, por exemplo, até 50 linhas na tabela:
%config SqlMagic.displaylimit = 10

## 1. Visão Geral das Tabelas e Contagem de Registros
Verificamos a integridade das tabelas criadas após o processo de ingestão.

In [2]:
query_contagem = """
SELECT 'customers' AS tabela, count(*) AS total_registros FROM customers
UNION ALL
SELECT 'orders', count(*) FROM orders
UNION ALL
SELECT 'order_items', count(*) FROM order_items
UNION ALL
SELECT 'order_payments', count(*) FROM order_payments
UNION ALL
SELECT 'order_reviews', count(*) FROM order_reviews
UNION ALL
SELECT 'products', count(*) FROM products
UNION ALL
SELECT 'sellers', count(*) FROM sellers;
"""
con.execute(query_contagem).df()

,tabela,total_registros
0,customers,99441
1,orders,99441
2,order_items,112650
3,order_payments,103886
4,order_reviews,99224
5,products,32951
6,sellers,3095


## Olist Brazilian E-Commerce: "Why are customers leaving bad reviews?

Our teams noticed that customer satisfaction has been decreasing and our team thinks it was caused due to the late deliveries. to answer that we need to do a set of queries to find out.

### What percentage of orders are delivered later than their estimated delivery date?

In [3]:
%%sql
--to answer this question we´ll be using the CASE.
--the CASE function allows us to filter when a delivery was a delay or not using date arithmetic
WITH orders_relation AS (
    SELECT
        CASE
            WHEN DATE_PART('day', order_estimated_delivery_date - order_delivered_customer_date) < 0 THEN 'atrasado'
            ELSE 'no_prazo_ou_adiantado'
        END AS status_entrega
    FROM orders
    WHERE order_status = 'delivered' 
      AND order_delivered_customer_date IS NOT NULL 
      AND order_estimated_delivery_date IS NOT NULL
)

/************************************************************************************************
after that we need to sum all the delayed orders.
we need to sum where the status_entrea = delayed/not_delayed, if it´s THEN 1 if not THEN 0
but when we devide an integer by another integer sql omites the decimal value letting a inprecise
number, so to solve that we use the cast operator ::decimal it converts the number on its left
turning it into a decimal value.
*************************************************************************************************/

SELECT
    (SUM(CASE WHEN status_entrega = 'atrasado' THEN 1 ELSE 0 END)::decimal / COUNT(*)) * 100 AS pct_atrasados,
    (SUM(CASE WHEN status_entrega = 'no_prazo_ou_adiantado' THEN 1 ELSE 0 END)::decimal / COUNT(*)) * 100 AS pct_entregues_no_prazo
FROM orders_relation;


Running query in 'DuckDBPyConnection'

pct_atrasados,pct_entregues_no_prazo
6.7730900798175595,93.22690992018245


> **Conclusão:** 
> Apenas **6,77%** das encomendas foram entregues com atraso. Isto indica que as encomendas atrasadas representam um grupo reduzido e que outros fatores podem estar a influenciar a insatisfação geral dos clientes.

## Is there a relationship between delivery delay and review score? (e.g., average review score for on-time vs. late orders)

In [4]:
%%sql
WITH delay_infor AS (
    SELECT
        CASE
            WHEN DATE_PART('day', order_estimated_delivery_date - order_delivered_customer_date) < 0 THEN 'delayed'
            ELSE 'delivered_on_time'
        END AS delivery_status,
        review_score
    FROM orders o 
    JOIN order_reviews os
        ON o.order_id = os.order_id
    WHERE order_status == 'delivered'
        AND order_estimated_delivery_date IS NOT NULL
        AND order_delivered_customer_date IS NOT NULL
)

SELECT
    delivery_status,
    AVG(review_score)
FROM delay_infor
GROUP BY delivery_status;


Running query in 'DuckDBPyConnection'

delivery_status,avg(review_score)
delivered_on_time,4.289980432268967
delayed,2.271025120923701


The analysis confirms the team’s hypothesis: delivery delays are directly linked to falling satisfaction. Customers who had received on delay has given reviews on average of 2.27 while customers that received on time has given an avg of 4.28

that means that delays has a significant impact on customers satisfaction almost cutting the score by half. With bigger delays the less will be the rating score.

If the marketplace dont enhance the logistics, the reputation and customers retention could get in risk